In [1]:
from pyspark.sql import functions as F
import com.microsoft.spark.fabric

air_quality = spark.table("lh_silver.dbo.air_quality")
zone_map = spark.table("lh_silver.dbo.zone_station_map")

joined = air_quality.join(zone_map, on=["station_id", "parameter"], how="inner")

cols = ["value", "station_id", "dist_km", "is_far_station",
        "is_low_coverage", "is_missing_value", "is_negative_value"]

aggs = [
    F.max(F.when(F.col("parameter") == p, F.col(c))).alias(f"{p}_{c}")
    for p in ("pm25", "no2")
    for c in cols
]

fact = joined.groupBy("zone_id", "date_local").agg(*aggs)
fact.write.mode("overwrite").synapsesql("wh_gold.dbo.fact_air_quality_daily")

g = spark.read.synapsesql("wh_gold.dbo.fact_air_quality_daily")
print(joined.filter(F.col("parameter") == "pm25").count(),
      g.filter(F.col("pm25_station_id").isNotNull()).count())
print(joined.filter(F.col("parameter") == "no2").count(),
      g.filter(F.col("no2_station_id").isNotNull()).count())


StatementMeta(, 9e1b4db9-5bbd-41ef-a14d-a2adbfba73d5, 3, Finished, Available, Finished, False)

787707 787707
872986 872986
28700 28700 28406
